# WDM Benchmarking and Accuracy Notebook

This notebook is a thin Colab wrapper around `docs/studies/benchmarks/colab_benchmarks.py`.

It benchmarks:
- NumPy runtime
- JAX CPU runtime
- JAX GPU runtime when a GPU runtime is available
- CuPy runtime when CuPy is installed
- 3-channel serial vs vectorized transforms
- round-trip reconstruction accuracy


## 1. Clone the repository into the Colab runtime

If you want to benchmark a branch or fork, replace the Git URL below before running the cell.

In [ ]:
import os
from pathlib import Path

repo_url = "https://github.com/pywavelet/wdm_transform.git"
repo_dir = Path("/content/wdm_transform")

if not repo_dir.exists():
    !git clone {repo_url} {repo_dir}

%cd /content/wdm_transform

## 2. Install the local package and optional accelerator backends

For GPU benchmarking in Colab, switch the runtime to GPU before running this cell.

The JAX and CuPy installs are best-effort. If one of them fails, the benchmark script will skip that backend and record the reason.

In [ ]:
%pip install -q -U pip setuptools wheel
%pip install -q -e .

try:
    get_ipython().system("pip install -q 'jax[cuda12]'")
except Exception as exc:
    print(f"JAX CUDA install skipped: {exc}")

try:
    get_ipython().system("pip install -q cupy-cuda12x")
except Exception as exc:
    print(f"CuPy install skipped: {exc}")

## 3. Run the benchmark script

This writes `colab_benchmark_results.json` plus one raw JSON file per backend that ran successfully.

In [ ]:
%run docs/studies/benchmarks/colab_benchmarks.py

## 4. Download the aggregate report

In [ ]:
from google.colab import files

files.download("colab_benchmark_results.json")